# 09. Paoay: features and crop predictions

Apply the saved three-crop model to matching satellite inputs. Preserve all raw scores before any screening. No training, retrieval or field-label assignment.


## 1. Setup

Finish combined 08 first. Use the unique saved model or set an explicit manifest. Clear current stage selections before starting so incomplete inputs cannot select earlier predictions.


In [ ]:
from pathlib import Path
import json
import sys

repository_root = next((folder for folder in [Path.cwd(), *Path.cwd().parents]
                        if (folder / "pyproject.toml").is_file() and (folder / "src/crop_pipeline").is_dir()), None)
if repository_root is None:
    raise FileNotFoundError("Open this notebook from inside the project repository")
if str(repository_root / "src") not in sys.path:
    sys.path.insert(0, str(repository_root / "src"))
from crop_pipeline.paoay import features, predictions, land_cover, dynamic_world
from crop_pipeline.analysis_workflow import resolve_stage_pointer, write_stage_pointer
from crop_pipeline.notebook_display import show_table
from crop_pipeline import progress as pipeline_progress

settings_path = repository_root / "settings.local.json"
local_settings = json.loads(settings_path.read_text()) if settings_path.exists() else {}
def local_path(value):
    path = Path(value).expanduser()
    return path.resolve() if path.is_absolute() else (repository_root / path).resolve()

model_manifest = None  # Optional explicit path to notebook 06's model_manifest.json.
report_folder = local_path(local_settings.get("report_root", "./sen4agrinet_work/inventory"))
paoay_root = report_folder.parent / "paoay"
guidance_pointer = paoay_root / "land_cover/land_cover_stage.json"
feature_root = paoay_root / "features"
feature_pointer = feature_root / "feature_stage.json"
prediction_root = paoay_root / "predictions"
prediction_pointer = prediction_root / "prediction_stage.json"
dynamic_world_pointer = paoay_root / "dynamic_world/dynamic_world_stage.json"
imagery_pointer = paoay_root / "sentinel2/imagery_stage.json"
chunk_size = 4096
for pointer in (guidance_pointer, feature_pointer, prediction_pointer):
    write_stage_pointer(pointer, status="pending", reason="Notebook 09 model application is incomplete")
stages = ["Verify model and satellite inputs", "Calculate exact model inputs", "Review feature readiness", "Apply fixed model", "Verify outputs", "Summary and next action"]
notebook_progress = pipeline_progress.start_notebook("09_paoay_model_application", stages, report_folder.parent / "progress", {"model": str(model_manifest)})


## 2. Verify model and inputs

Retain the existing guidance dependency for cache compatibility; it is provenance only and never excludes feature pixels. The scientific screening review happens in 10. Exact feature order, formula recipe, dates and quality requirements remain unchanged.


In [ ]:
pipeline_progress.begin_stage("Verify model and satellite inputs")
model_path = features.find_model(report_folder.parent / "analysis", local_path(model_manifest) if model_manifest else None)
imagery_handoff = resolve_stage_pointer(imagery_pointer)
dynamic_world_handoff = resolve_stage_pointer(dynamic_world_pointer)
world_bundle = dynamic_world.load_dynamic_world_bundle(dynamic_world_handoff)
if Path(world_bundle["manifest"]["imagery_handoff"]).resolve() != Path(imagery_handoff).resolve():
    raise ValueError("Satellite sources do not match; finish combined notebook 08")
# Compatibility provenance only: retain existing feature identities and cache reuse.
# No guidance class filters any crop-model input.
guidance_request = land_cover.prepare_request(dynamic_world_handoff, paoay_root / "land_cover", land_cover.DEFAULT_RULES)
guidance_handoff = land_cover.build(guidance_request)
request = features.prepare_request(guidance_handoff, model_path, feature_root)
recipe = features.load_model_recipe(model_path)["recipe"]
print("Selected fixed model:", model_path)
print("Feature design:", recipe["selected_design"], "— ordered predictors:", len(recipe["feature_columns"]))
print("Minimum per-composite clear coverage:", recipe["quality"]["minimum_valid_fraction"])
print("Land-cover filtering: no. Model refitting: no.")
pipeline_progress.end_stage()


## 3. Calculate features

Reuse verified bundles when possible. Otherwise calculate the saved recipe with one pixel-progress bar. All eight bins must pass the original coverage and finite-value checks.


In [ ]:
pipeline_progress.begin_stage("Calculate exact model inputs")
def pixel_progress(items):
    return pipeline_progress.track(items, "Calculate crop-model features", unit="pixel")
feature_handoff = features.build(request, iterator=pixel_progress)
bundle = features.load_feature_bundle(feature_handoff, require_ready=False)
pipeline_progress.end_stage()


## 4. Check readiness

Stop on zero usable rows; keep audits and leave stage selections pending. No imputation or relaxed quality thresholds.


In [ ]:
pipeline_progress.begin_stage("Review feature readiness")
show_table(bundle["coverage"], title="Composite coverage inside the ROI")
show_table(bundle["audit"], title="Pixel readiness and exclusions")
print("ROI pixels:", bundle["manifest"]["roi_pixels"])
print("Ready feature rows:", bundle["manifest"]["row_count"])
print("Matrix shape:", bundle["matrix"].shape)
print("Land-cover filtering applied: no")
if bundle["manifest"]["status"] != "prepared":
    raise RuntimeError("No ready feature pixels; review the coverage and pixel audit above. All current stage selections remain pending.")
pipeline_progress.end_stage()


## 5. Predict

Apply the fixed classifier in computational batches. Other crop species remain possible within the boundaries; confidence cannot reliably detect them.


In [ ]:
pipeline_progress.begin_stage("Apply fixed model")
def inference_progress(items):
    return pipeline_progress.track(items, "Apply saved crop model", unit="batch")
prediction_request = predictions.prepare_request(feature_handoff, prediction_root)
prediction_handoff = predictions.build(prediction_request, chunk_size=chunk_size, iterator=inference_progress)
bundle = predictions.load_prediction_bundle(prediction_handoff)
pipeline_progress.end_stage()


## 6. Verify and publish

Check matching dependencies and saved scores, then publish the existing handoff formats. Areas below describe raw model choices, not verified planted area.


In [ ]:
pipeline_progress.begin_stage("Verify outputs")
show_table([{**row, "code": int(row["code"]), "pixels": int(row["pixels"]), "area_ha": float(row["area_ha"]),
             "percent_of_predicted_pixels": float(row["percent_of_predicted_pixels"])} for row in bundle["summary"]],
           title="Crop-only predicted areas")
manifest = bundle["manifest"]
print("ROI pixels:", manifest["roi_pixels"])
print("Predicted pixels:", manifest["row_count"])
print("ROI pixels without valid crop inputs:", manifest["roi_pixels"] - manifest["row_count"])
print("Tied top probabilities:", manifest["tied_pixels"])
print("Classifier class order:", ", ".join(manifest["classifier_class_order"]))
print("Class codes:", "; ".join(f"{name} = {code}" for name, code in manifest["class_codes"].items()))
features.load_feature_bundle(feature_handoff)
predictions.load_prediction_bundle(prediction_handoff)
write_stage_pointer(guidance_pointer, guidance_handoff)
write_stage_pointer(feature_pointer, feature_handoff)
write_stage_pointer(prediction_pointer, prediction_handoff)
pipeline_progress.end_stage()


## 7. Next step

Unknown and screening flags belong to 10. Alternative other-crop training methods remain TODOs.


In [ ]:
pipeline_progress.begin_stage("Summary and next action")
print("Verified features:", feature_handoff)
print("Verified raw predictions:", prediction_handoff)
print("These are forced choices among three crops, including on other crop species and noncrop surfaces.")
print("Next: notebook 10 creates boundary-limited maps, Unknown and separate screening flags.")
print("Model, training partitions and imagery preserved; no fitting or downloads.")
print("Detail log:", notebook_progress.output.path)
pipeline_progress.end_stage()
pipeline_progress.finish_notebook()
